## Inspection summary

- The training and validation files have the same structure: 529 columns.
- The first 520 columns are Wi-Fi access-point signal readings.
- No true missing values were found.
- A value of 100 in a WAP column means that the access point was not detected.
- BUILDINGID and FLOOR will be classification targets.
- LONGITUDE and LATITUDE will be regression targets.
- Some target classes, especially floor 4, have fewer observations than others.

In [4]:
for column in ["LONGITUDE", "LATITUDE"]:
    q1 = train_df[column].quantile(0.25)
    q3 = train_df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (train_df[column] < lower_bound) |
        (train_df[column] > upper_bound)
    ).sum()

    print(f"{column}")
    print(f"  IQR bounds: {lower_bound:.2f} to {upper_bound:.2f}")
    print(f"  Potential outliers: {outlier_count}")

LONGITUDE
  IQR bounds: -7948.05 to -7005.88
  Potential outliers: 0
LATITUDE
  IQR bounds: 4864657.35 to 4865093.41
  Potential outliers: 0


In [2]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

train_df = pd.read_csv(data_dir / "trainingData.csv")
validation_df = pd.read_csv(data_dir / "validationData.csv")

In [3]:
train_df[["LONGITUDE", "LATITUDE", "FLOOR", "BUILDINGID"]].corr()

,LONGITUDE,LATITUDE,FLOOR,BUILDINGID
LONGITUDE,1.000000,-0.860326,0.080836,0.958374
LATITUDE,-0.860326,1.000000,-0.101685,-0.881269
FLOOR,0.080836,-0.101685,1.000000,0.114734
BUILDINGID,0.958374,-0.881269,0.114734,1.000000


### Correlation and outlier checks

Longitude and latitude are strongly negatively correlated because they describe positions within the surveyed indoor area. Building ID is also associated with coordinate ranges, although it is a categorical code and should not be interpreted as a causal numeric variable.

Using the 1.5 × IQR rule, no potential outliers were identified for either longitude or latitude. Therefore, no coordinate rows were removed.

In [7]:
print(validation_df["FLOOR"].value_counts().sort_index().to_string())

FLOOR
0    132
1    462
2    306
3    172
4     39


In [6]:
print("Training — BuildingID distribution:")
print(train_df["BUILDINGID"].value_counts().sort_index())

print("\nTraining — FLOOR distribution:")
print(train_df["FLOOR"].value_counts().sort_index())

print("\nValidation — BuildingID distribution:")
print(validation_df["BUILDINGID"].value_counts().sort_index())

print("\nValidation — FLOOR distribution:")
print(validation_df["FLOOR"].value_counts().sort_index())

Training — BuildingID distribution:
BUILDINGID
0    5249
1    5196
2    9492
Name: count, dtype: int64

Training — FLOOR distribution:
FLOOR
0    4369
1    5002
2    4416
3    5048
4    1102
Name: count, dtype: int64

Validation — BuildingID distribution:
BUILDINGID
0    536
1    307
2    268
Name: count, dtype: int64

Validation — FLOOR distribution:
FLOOR
0    132
1    462
2    306
3    172
4     39
Name: count, dtype: int64


In [5]:
print("Data types in the training dataset:")
print(train_df.dtypes.value_counts())

print("\nKey target-column data types:")
print(train_df[["LONGITUDE", "LATITUDE", "FLOOR", "BUILDINGID"]].dtypes)

print("\nTotal missing values:")
print("Training:", train_df.isna().sum().sum())
print("Validation:", validation_df.isna().sum().sum())

Data types in the training dataset:
int64      527
float64      2
Name: count, dtype: int64

Key target-column data types:
LONGITUDE     float64
LATITUDE      float64
FLOOR           int64
BUILDINGID      int64
dtype: object

Total missing values:
Training: 0
Validation: 0


In [4]:
print("First 10 columns:")
print(train_df.columns[:10].tolist())

print("\nLast 9 columns:")
print(train_df.columns[-9:].tolist())

print("\nTraining and validation columns match:")
print(train_df.columns.equals(validation_df.columns))

wap_columns = [column for column in train_df.columns if column.startswith("WAP")]
print("\nNumber of Wi-Fi feature columns:", len(wap_columns))

First 10 columns:
['WAP001', 'WAP002', 'WAP003', 'WAP004', 'WAP005', 'WAP006', 'WAP007', 'WAP008', 'WAP009', 'WAP010']

Last 9 columns:
['LONGITUDE', 'LATITUDE', 'FLOOR', 'BUILDINGID', 'SPACEID', 'RELATIVEPOSITION', 'USERID', 'PHONEID', 'TIMESTAMP']

Training and validation columns match:
True

Number of Wi-Fi feature columns: 520


In [3]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Training shape: (19937, 529)
Validation shape: (1111, 529)


In [2]:
data_dir = Path("data")

train_path = data_dir / "trainingData.csv"
validation_path = data_dir / "validationData.csv"

print("Training file exists:", train_path.exists())
print("Validation file exists:", validation_path.exists())

Training file exists: True
Validation file exists: True


In [1]:
import pandas as pd
from pathlib import Path